# The "1.58-Bit" Inference Engine — Custom Triton Kernels

BitNet b1.58 stores every weight as a ternary value `{-1, 0, 1}` ("1.58 bit" = log2(3)) plus one scale `gamma`.
Multiplying by such a weight is just **add / subtract / skip** — no multiplier needed.

A T4 has no native 2-bit math, so the real win is **memory bandwidth**: each weight moves as **2 bits** instead of
fp16's 16 (8x less weight traffic). During decode (batch=1) the matmul is a memory-bound GEMV dominated by reading
the weight matrix, so this directly raises tokens/sec. This notebook builds the kernel, proves correctness, and
benchmarks it — honestly reporting where it wins (decode) and where cuBLAS still wins (large-batch prefill).

**Setup:** Kaggle → Settings → Accelerator **GPU T4 x2**, Internet **On**.

## 1. Get the code & install deps
Kaggle's GPU image ships CUDA torch + Triton; we only add `transformers`.

In [ ]:
import os, sys
REPO = '1.58-Bit-Inference-Engine-Custom-Triton-Kernels-'
if not os.path.exists(REPO):
    !git clone https://github.com/aabhimittal/1.58-bit-inference-engine-custom-triton-kernels-.git {REPO}
sys.path.insert(0, os.path.abspath(REPO))
!pip install -q transformers accelerate

In [ ]:
import torch, triton
print('torch', torch.__version__, '| triton', triton.__version__)
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Quantize & pack: ternary weights into int8
Absmean quantization gives `{-1,0,1}` + a scale; four ternary codes pack into one byte.

In [ ]:
from bitnet.quantize import absmean_quantize, pack_ternary, unpack_ternary

W = torch.randn(8, 16)
ternary, gamma = absmean_quantize(W)
packed, k_pad = pack_ternary(ternary)
print('weight  :', tuple(W.shape), W.dtype, '->', W.numel()*2, 'bytes (fp16)')
print('packed  :', tuple(packed.shape), packed.dtype, '->', packed.numel(), 'bytes (2-bit)')
print('compression vs fp16:', round(W.numel()*2 / packed.numel(), 2), 'x')
assert torch.equal(unpack_ternary(packed, k=W.shape[1]), ternary)  # exact round-trip
print('round-trip OK, unique ternary values:', ternary.unique().tolist())

## 3. The custom Triton kernel
`bitnet_matmul` dispatches to an **addition-only GEMV** for small M (decode) and a `tl.dot` GEMM for large M
(prefill). Let's verify it matches the dequantized reference.

In [ ]:
from kernels.bitnet_kernel import bitnet_matmul, ternary_matmul_reference

N, K = 4096, 4096
ternary, gamma = absmean_quantize(torch.randn(N, K))
packed = pack_ternary(ternary)[0].cuda(); gamma = gamma.cuda()
x = torch.randn(1, K, device='cuda', dtype=torch.float16)  # decode: M=1

y   = bitnet_matmul(x, packed, gamma, k=K)
ref = ternary_matmul_reference(x, packed, gamma, k=K)
print('max abs error vs reference:', (y.float()-ref.float()).abs().max().item())

## 4. Microbenchmark: 1.58-bit kernel vs fp16 cuBLAS
The decode path (M=1) is where 2-bit weights win on bandwidth.

In [ ]:
!python {REPO}/benchmark/microbench.py

## 5. End-to-end on a real BitNet model
Load a small BitNet checkpoint, measure baseline tokens/sec, swap every projection for `BitLinear`
(packed-ternary + Triton kernel), and measure again.

In [ ]:
from bitnet.model_utils import load_model, replace_linears, tokens_per_second

MODEL_ID = '1bitLLM/bitnet_b1_58-large'   # or 'microsoft/bitnet-b1.58-2B-4T'
model, tok = load_model(MODEL_ID)
base = tokens_per_second(model, tok, max_new_tokens=128)
print(f'baseline (fp16 Linear): {base:.2f} tok/s')

In [ ]:
n = replace_linears(model)
print(f'swapped {n} Linear layers -> packed-ternary BitLinear')
bit = tokens_per_second(model, tok, max_new_tokens=128)
print(f'custom (1.58-bit kernel): {bit:.2f} tok/s   |   speedup: {bit/base:.2f}x')

## 6. Takeaways
- Ternary weights pack 8x smaller than fp16; the multiply is add/subtract/skip.
- The Triton kernel turns that into a real **decode-time tokens/sec** gain on a T4 by removing the weight-bandwidth bottleneck.
- For large-batch prefill, cuBLAS tensor cores can still win — expected, since the T4 has no native 2-bit FLOPs.
- Paste your numbers into the README results table.